In [1]:
import pandas as pd

In [3]:
# 1. Load both prediction files
rule_preds = pd.read_csv('data/processed/abt_buy_rulebased_preds.csv')
llm_preds = pd.read_csv('data/processed/abt_buy_llm_zeroshot_preds.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/abt_buy_rulebased_preds.csv'

In [12]:
# ---- 2. Merge on id_abt, id_buy ----
# rule_preds has: id_abt, id_buy, label, pred_score, pred_label
# llm_preds has: id_abt, id_buy, name_abt, name_buy, label, raw_response, pred_label
merged = rule_preds[['id_abt', 'id_buy', 'label', 'pred_label']].merge(
    llm_preds[['id_abt', 'id_buy', 'name_abt', 'name_buy', 'pred_label', 'raw_response']],
    on=['id_abt', 'id_buy'],
    suffixes=('_rule', '_llm')
)

print(f"Merged rows: {len(merged)}")  # sanity check — should be 2194

Merged rows: 2194


In [15]:
#3. Categorize each pair
def categorize(row):
    rule_correct = row['pred_label_rule'] == row['label']
    llm_correct = row['pred_label_llm'] == row['label']

    if rule_correct and llm_correct:
        return 'both_correct'
    elif not rule_correct and not llm_correct:
        return 'both_wrong'
    elif llm_correct and not rule_correct:
        return 'llm_right_rule_wrong'
    else:  # rule_correct and not llm_correct
        return 'rule_right_llm_wrong'

merged['category'] = merged.apply(categorize, axis=1)

In [17]:
# ---- 4. Print category counts ----
print("\nCategory counts:")
print(merged['category'].value_counts())


Category counts:
category
both_correct            2022
rule_right_llm_wrong      67
llm_right_rule_wrong      64
both_wrong                41
Name: count, dtype: int64


In [18]:
# ---- 5. Pull out disagreement cases ----
llm_right_rule_wrong = merged[merged['category'] == 'llm_right_rule_wrong']
rule_right_llm_wrong = merged[merged['category'] == 'rule_right_llm_wrong']

pd.set_option('display.max_colwidth', 60)

print(f"\n--- LLM right, rule-based wrong ({len(llm_right_rule_wrong)} cases) ---")
print(llm_right_rule_wrong[['name_abt', 'name_buy', 'label', 'pred_label_rule', 'pred_label_llm']].head(10))

print(f"\n--- Rule-based right, LLM wrong ({len(rule_right_llm_wrong)} cases) ---")
print(rule_right_llm_wrong[['name_abt', 'name_buy', 'label', 'pred_label_rule', 'pred_label_llm']].head(10))


--- LLM right, rule-based wrong (64 cases) ---
                                                        name_abt  \
5          iHome Black Clock Radio Audio System For iPod - IH9BR   
14           Hoover Bagged Tempo Widepath Upright Vacuum - U5140   
55        Sanus Black Single-Column AV Component System - VF2012   
82   Panasonic 2 Line 5.8 GHz FHSS GigaRange Expandable Digit...   
133              Belkin Cush Top For Computer Laptop - F8N044ORG   
161                 Sony Black Active Speaker System - SRSA212BK   
190  Panasonic Integrated Black Telephone System With All-Dig...   
249             Sony Active Style Headphones In Black - MDRAS50G   
269      iHome iPod & iPhone Clock Radio & Audio System - IP99BR   
292  Sanus 15' - 37' VisionMount Full-Motion Flat Panel TV Bl...   

                                                        name_buy  label  \
5    IH9B6R BLACK ALARM CLOCK F/IPODPERPCHARGES DOCKED IPOD R...      1   
14                 Hoover Company #U5140900 Wide Path

In [19]:
# ---- 6. Save disagreement cases for your report/notes ----
llm_right_rule_wrong.to_csv('data/processed/disagree_llm_right.csv', index=False)
rule_right_llm_wrong.to_csv('data/processed/disagree_rule_right.csv', index=False)
print("\nSaved disagreement cases to data/processed/")


Saved disagreement cases to data/processed/


In [20]:
# ---- 7. Also look at 'both_wrong' — these are genuinely hard/ambiguous cases ----
both_wrong = merged[merged['category'] == 'both_wrong']
print(f"\n--- Both wrong ({len(both_wrong)} cases) — worth a look too ---")
print(both_wrong[['name_abt', 'name_buy', 'label', 'pred_label_rule', 'pred_label_llm']].head(10))


--- Both wrong (41 cases) — worth a look too ---
                                                        name_abt  \
29                                 Canon Color Ink Tank - CL41CL   
153                 BlueAnt Bluetooth Voice Control Headset - V1   
181                 Canon Silver Flash Memory Camcorder - FS100S   
208  LG 25.0 Cu. Ft. Titanium French Door Bottom Freezer Refr...   
264  Garmin Nuvi 360 010-10723-06 Black 12 Volt Adapter Cable...   
382  LG 24' LDF6920WW Fully Integrated Built In White Dishwas...   
448  Electrolux Pronto 2 In 1 Lightweight Upright Vacuum - EL...   
458  Sony VAIO Neoprene Laptop Carrying Case - Black Finish -...   
524   Panasonic 27' Stainless Steel Microwave Trim Kit - TK903SS   
556                      Canon Black Photo Ink Cartridge - CLI8B   

                                                        name_buy  label  \
29   Canon Ink Cartridge For PIXMA iP1600, iP6210D and iP6220...      1   
153                   BLUEANT BLUETOOTH HS DUAL MIC

In [21]:
rule_right_llm_wrong

,id_abt,id_buy,label,pred_label_rule,name_abt,name_buy,pred_label_llm,raw_response,category
8,35010,208117938,1,1,LG 30' White Freestanding Gas Range - LRG30357WH,LG 5.0 cu.ft. Freestanding Gas Range,0,NO_MATCH,rule_right_llm_wrong
89,32022,204363358,1,1,Sony Bluetooth Adaptor/Portable Transmitter - TMRBT10,SONY STEREO BLUETOOTH HEADSET *NIC* - TMRBT10/TMRBT10A,0,NO_MATCH,rule_right_llm_wrong
99,36299,208342524,1,1,Griffin Black iPhone 3G Wave Case - 8227IP2WVB,Griffin Protective Wave Case for Smart Phone - 8227-IP2WVB,0,NO_MATCH,rule_right_llm_wrong
131,36083,206362623,1,1,Canon Deluxe Grey Leather Case - 2349B001,Canon PSC-1000 Semi-Hard Leather Case - 2349B001,0,NO_MATCH,rule_right_llm_wrong
135,38399,210479062,1,1,Canon 2GB SD Secure Digital Card - 3505B001,Simpletech 2GB SD Card - 3505B001AA,0,NO_MATCH,rule_right_llm_wrong
...,...,...,...,...,...,...,...,...,...
2102,35903,205769439,1,1,Sony 2GB Memory Stick Micro (M2) - MSA2GU2,Sony 2GB Memory Stick Micro (M2) - MSA2GD,0,NO_MATCH,rule_right_llm_wrong
2134,32228,205985718,1,1,LG DLEX8377NM Navy Blue XL Capacity Electric SteamDryer ...,The LG Electric SteamDryer,0,NO_MATCH,rule_right_llm_wrong
2141,33532,204071217,1,1,TomTom GPS Mount And USB Car Charger - 9N00101,TOMTOM GPS Receiver Accessory Kit - 9N00.101,0,NO_MATCH,rule_right_llm_wrong
2162,34927,208117933,1,1,LG Stainless Steel Freestanding Electric Range - LRE30757SS,LG 5.6 cu.ft. Freestanding Electric Range,0,NO_MATCH,rule_right_llm_wrong


In [22]:
both_wrong

,id_abt,id_buy,label,pred_label_rule,name_abt,name_buy,pred_label_llm,raw_response,category
29,20450,201692660,1,0,Canon Color Ink Tank - CL41CL,"Canon Ink Cartridge For PIXMA iP1600, iP6210D and iP6220...",0,NO_MATCH,both_wrong
153,37589,209085184,1,0,BlueAnt Bluetooth Voice Control Headset - V1,BLUEANT BLUETOOTH HS DUAL MIC NIC - 091004,0,NO_MATCH,both_wrong
181,33277,206808445,1,0,Canon Silver Flash Memory Camcorder - FS100S,Canon FS100 Digital Camera - 2699B001,0,NO_MATCH,both_wrong
208,34948,208114675,1,0,LG 25.0 Cu. Ft. Titanium French Door Bottom Freezer Refr...,LG 25.0 Cu.Ft. Total Capacity,0,NO_MATCH,both_wrong
264,25857,202870177,1,0,Garmin Nuvi 360 010-10723-06 Black 12 Volt Adapter Cable...,Garmin Cigarette Lighter Adapter for GPS,0,NO_MATCH,both_wrong
382,35183,208117920,1,0,LG 24' LDF6920WW Fully Integrated Built In White Dishwas...,LG Dishwasher,0,NO_MATCH,both_wrong
448,30058,209028436,1,0,Electrolux Pronto 2 In 1 Lightweight Upright Vacuum - EL...,ELECTROLUX STICK VACUUM REPLACES EL1000A,0,NO_MATCH,both_wrong
458,20448,10377226,1,0,Sony VAIO Neoprene Laptop Carrying Case - Black Finish -...,Sony Notebook and AC Adapter Cases - VGPAMC3,0,NO_MATCH,both_wrong
524,32027,210461432,1,0,Panasonic 27' Stainless Steel Microwave Trim Kit - TK903SS,PAN NN-TK903S MICROWAVE OVEN .08 CU. FT.,0,NO_MATCH,both_wrong
556,33047,201692665,1,0,Canon Black Photo Ink Cartridge - CLI8B,"Canon Ink Cartridge For PIXMA iP4200, iP5200, iP5200R an...",0,NO_MATCH,both_wrong
